In [ ]:
import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.model_selection import train_test_split

CFG = {
    'D': 256,
    'H': 4,
    'N': 32768,
    'VOCAB': 50257,
    'RETRIEVAL_TOP_K': 3,
    'EPOCHS': 15,
    'DEVICE': torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    'MAX_LEN': 100
}

print(f"Running on {CFG['DEVICE']}")

def find_file(filename_part):
    for root, dirs, files in os.walk('/kaggle/input'):
        for name in files:
            if filename_part in name: return os.path.join(root, name)
    if os.path.exists(filename_part): return filename_part
    return None

train_path = find_file('train (2).csv') or find_file('train.csv')
test_path = find_file('test (2).csv') or find_file('test.csv')
novels = [find_file('The Count of Monte Cristo.txt'), find_file('In search of the castaways.txt')]
novels = [n for n in novels if n]

class SimpleRetriever:
    def __init__(self, novel_paths):
        self.chunks = []
        print("Indexing Novels for Retrieval...")
        for path in novel_paths:
            with open(path, 'r', encoding='utf-8') as f: text = f.read()
            words = text.split()
            chunk_size = 150
            for i in range(0, len(words), chunk_size):
                chunk = " ".join(words[i:i+chunk_size])
                self.chunks.append(chunk)
        
        self.vectorizer = TfidfVectorizer(stop_words='english', max_features=50000)
        self.tfidf_matrix = self.vectorizer.fit_transform(self.chunks)
        print(f"Indexed {len(self.chunks)} chunks.")

    def retrieve(self, query, top_k=3):
        try:
            query_vec = self.vectorizer.transform([query])
            scores = (self.tfidf_matrix * query_vec.T).toarray().flatten()
            top_indices = scores.argsort()[-top_k:][::-1]
            return [self.chunks[i] for i in top_indices if scores[i] > 0]
        except:
            return []

retriever = SimpleRetriever(novels)

class BDH_Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.wte = nn.Embedding(CFG['VOCAB'], CFG['D'])
        self.ln = nn.LayerNorm(CFG['D'])
        self.decoder = nn.Parameter(torch.randn(CFG['H'], CFG['D'], CFG['N'] // CFG['H']) * 0.02)

    def forward(self, idx):
        x = self.ln(self.wte(idx))
        return x

class DualStreamMemory(nn.Module):
    def __init__(self):
        super().__init__()
        self.global_state = nn.Parameter(torch.randn(CFG['N'], CFG['D']) * 0.01)
        
        self.gate_net = nn.Sequential(nn.Linear(CFG['D']*2, CFG['D']), nn.ReLU(), nn.Linear(CFG['D'], 1), nn.Sigmoid())
        self.forget_net = nn.Sequential(nn.Linear(CFG['D']*2, 1), nn.Sigmoid())
        
        self.recon_head = nn.Linear(CFG['D'], CFG['D'])
        self.wm_proj = nn.Linear(CFG['D'], CFG['D'])

    def update_global(self, e_id, attr_vec):
        current = self.global_state[e_id].unsqueeze(0)
        
        combined = torch.cat([attr_vec, current], dim=-1)
        i_gate = self.gate_net(combined)
        f_gate = self.forget_net(combined)
        
        new_mem = (f_gate * current) + (i_gate * attr_vec)
        return new_mem, i_gate, f_gate, self.recon_head(new_mem)

    def probe(self, char_id, backstory_vec, retrieved_vecs):
        global_trace = self.recon_head(self.global_state[char_id].unsqueeze(0))
        sim_global = F.cosine_similarity(global_trace, backstory_vec)
        
        if retrieved_vecs is not None and len(retrieved_vecs) > 0:
            wm_trace = self.wm_proj(torch.mean(retrieved_vecs, dim=0, keepdim=True))
            sim_local = F.cosine_similarity(wm_trace, backstory_vec)
        else:
            sim_local = sim_global 
            
        return sim_global, sim_local

def validate(model_enc, model_mem, val_df, retriever):
    model_enc.eval(); model_mem.eval()
    scores, labels = [], []
    label_map = {'consistent': 1, 'contradict': 0}
    
    with torch.no_grad():
        for _, row in val_df.iterrows():
            b_words = str(row['content']).split()
            if not b_words: continue
            
            b_toks = torch.tensor([[hash(w)%CFG['VOCAB'] for w in b_words]], device=CFG['DEVICE'])
            b_vec = torch.mean(model_enc(b_toks), dim=1)
            
            chunks = retriever.retrieve(f"{row['char']} {row['content']}", top_k=2)
            ret_vecs = []
            for c in chunks:
                c_words = c.split()[:50]
                c_toks = torch.tensor([[hash(w)%CFG['VOCAB'] for w in c_words]], device=CFG['DEVICE'])
                rv = model_enc(c_toks)
                ret_vecs.append(torch.mean(rv, dim=1).squeeze(0))
            ret_tensor = torch.stack(ret_vecs) if ret_vecs else None
            
            eid = hash(row['char']) % CFG['N']
            sim_g, sim_l = model_mem.probe(eid, b_vec, ret_tensor)
            
            score = (0.4 * sim_g) + (0.6 * sim_l)
            scores.append(score.item())
            labels.append(label_map[row['label']])
    
    best_f1 = 0
    best_acc = 0
    for t in np.linspace(-0.5, 0.5, 50):
        preds = [1 if s >= t else 0 for s in scores]
        f1 = f1_score(labels, preds, average='weighted')
        if f1 > best_f1: 
            best_f1 = f1
            best_acc = balanced_accuracy_score(labels, preds)
            
    return best_f1, best_acc

def train_hybrid_hash(model_enc, model_mem, optimizer, train_df, val_df, novels):
    print(f"Starting Hash+TF-IDF Training ({CFG['EPOCHS']} Epochs)...")
    
    best_val_f1 = -1
    
    for epoch in range(CFG['EPOCHS']):
        model_enc.train(); model_mem.train()
        
        for novel in novels:
            with open(novel, 'r', encoding='utf-8') as f: text = f.read()
            chapters = re.split(r'(?im)^\s*Chapter\s+(?:[0-9]+|[IVXLCDM]+)\b', text)
            chapters = [c for c in chapters if len(c) > 500]
            
            for chapter in chapters[:15]: 
                words = chapter.split()[:100]
                for word in words:
                    optimizer.zero_grad()
                    tid = torch.tensor([[hash(word) % CFG['VOCAB']]], device=CFG['DEVICE'])
                    vec = model_enc(tid).squeeze(1)
                    
                    eid = hash(word) % CFG['N']
                    
                    new_mem, i, f, recon = model_mem.update_global(eid, vec)
                    model_mem.global_state.data[eid] = new_mem.detach()
                    
                    loss = F.mse_loss(recon, vec.detach()) + 0.1*torch.mean(i)
                    loss.backward()
                    optimizer.step()

        total_cons_loss = 0
        batch = train_df.sample(min(64, len(train_df)))
        optimizer.zero_grad()
        
        for _, row in batch.iterrows():
            b_words = str(row['content']).split()
            if not b_words: continue
            
            b_toks = torch.tensor([[hash(w)%CFG['VOCAB'] for w in b_words]], device=CFG['DEVICE'])
            b_vec = torch.mean(model_enc(b_toks), dim=1)
            
            chunks = retriever.retrieve(f"{row['char']} {row['content']}", top_k=2)
            ret_vecs = []
            for c in chunks:
                c_words = c.split()[:50]
                c_toks = torch.tensor([[hash(w)%CFG['VOCAB'] for w in c_words]], device=CFG['DEVICE'])
                rv = model_enc(c_toks)
                ret_vecs.append(torch.mean(rv, dim=1).squeeze(0))
            ret_tensor = torch.stack(ret_vecs) if ret_vecs else None
            
            eid = hash(row['char']) % CFG['N']
            sim_g, sim_l = model_mem.probe(eid, b_vec, ret_tensor)
            hybrid_sim = (sim_g + sim_l) / 2.0
            
            target = 1.0 if row['label'] == 'consistent' else -1.0
            loss = F.relu(0.5 - hybrid_sim) if target == 1 else F.relu(hybrid_sim + 0.5)
            total_cons_loss += loss

        total_cons_loss = total_cons_loss / len(batch)
        total_cons_loss.backward()
        optimizer.step()
        
        val_f1, val_acc = validate(model_enc, model_mem, val_df, retriever)
        print(f"Epoch {epoch+1:02d} | Loss: {total_cons_loss.item():.4f} | Val F1: {val_f1:.4f} | Val Bal Acc: {val_acc:.4f}")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save({'enc': model_enc.state_dict(), 'mem': model_mem.state_dict()}, "best_hybrid_model.pth")
            print("  >>> New Best Model Saved!")

def predict_dataset(model_enc, model_mem, df, is_test=True):
    print("Loading Best Model...")
    checkpoint = torch.load("best_hybrid_model.pth", map_location=CFG['DEVICE'])
    model_enc.load_state_dict(checkpoint['enc'])
    model_mem.load_state_dict(checkpoint['mem'])
    
    model_enc.eval(); model_mem.eval()
    preds, ids = [], []
    THRESHOLD = 0.05 
    
    print(f"Generating Predictions...")
    with torch.no_grad():
        for idx, row in tqdm(df.iterrows(), total=len(df)):
            ids.append(row['id'] if 'id' in row else idx)
            b_words = str(row['content']).split()
            if not b_words: 
                preds.append("consistent"); continue
            
            b_toks = torch.tensor([[hash(w)%CFG['VOCAB'] for w in b_words]], device=CFG['DEVICE'])
            b_vec = torch.mean(model_enc(b_toks), dim=1)
            
            chunks = retriever.retrieve(f"{row['char']} {row['content']}", top_k=3)
            ret_vecs = []
            for c in chunks:
                c_words = c.split()[:50]
                c_toks = torch.tensor([[hash(w)%CFG['VOCAB'] for w in c_words]], device=CFG['DEVICE'])
                rv = model_enc(c_toks)
                ret_vecs.append(torch.mean(rv, dim=1).squeeze(0))
            ret_tensor = torch.stack(ret_vecs) if ret_vecs else None
            
            eid = hash(row['char']) % CFG['N']
            sim_g, sim_l = model_mem.probe(eid, b_vec, ret_tensor)
            
            score = (0.4 * sim_g) + (0.6 * sim_l)
            if score < THRESHOLD: preds.append("contradict")
            else: preds.append("consistent")
            
    if is_test: return pd.DataFrame({'id': ids, 'prediction': preds})
    else: return pd.DataFrame({'id': ids, 'char': df['char'], 'content': df['content'], 'true_label': df['label'], 'predicted_label': preds})

encoder = BDH_Encoder().to(CFG['DEVICE'])
memory = DualStreamMemory().to(CFG['DEVICE'])
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(memory.parameters()), lr=1e-3)

df_full = pd.read_csv(train_path)
df_train, df_val = train_test_split(df_full, test_size=0.2, random_state=42, stratify=df_full['label'])
df_test = pd.read_csv(test_path)

print(f"Train Size: {len(df_train)} | Val Size: {len(df_val)}")

train_hybrid_hash(encoder, memory, optimizer, df_train, df_val, novels)

val_preds = predict_dataset(encoder, memory, df_val, is_test=False)
val_preds.to_csv("validation_predictions.csv", index=False)
print("Saved validation_predictions.csv")

test_preds = predict_dataset(encoder, memory, df_test, is_test=True)
test_preds.to_csv("results.csv", index=False)
print("Saved results.csv")
print(test_preds.head())